In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [7]:
## Read table
fname = 'vetting-v01'
fpath = f'mnt/tess/labels/{fname}.csv'

second_f=True

if second_f:
    fname_extra_objects = 'vetting-new-events-withfilenames'
    fpath_extra_objects=f'mnt/tess/labels/{fname_extra_objects}.csv'



# Read CSVs and set 'Astro ID' as index
all_table = pd.read_csv(fpath, index_col='Astro ID', low_memory=False)
print('Original table shape:', all_table.shape)

if second_f:
    long_per_table = pd.read_csv(fpath_extra_objects, index_col='Astro ID', low_memory=False)
    print('Long period table shape:', long_per_table.shape)

    all_table_2=pd.concat([all_table, long_per_table], ignore_index=False)

    all_table=all_table_2
    print('Table shape after adding long periods:', all_table.shape)

Original table shape: (9344, 32)
Long period table shape: (1647, 59)
Table shape after adding long periods: (10991, 60)


In [8]:
all_table.columns

Index(['TIC ID', 'Final', 'Decision', 'Distinct', 'mk', 'ch', 'et', 'md', 'as',
       'dm', 'Tansu', 'Shishir', 'jh', 'Astronet note',
       'Seed randbetween(1, 100)', 'RA', 'Dec', 'Tmag', 'Epoc', 'Period',
       'Duration', 'Transit_Depth', 'Sectors', 'star_rad', 'star_mass', 'teff',
       'logg', 'SN', 'Qingress', 'star_rad_est', 'filename', 'comment',
       'Parameter Source Pipeline', 'Detection Pipeline(s)', 'Full TOI ID',
       'TIC Right Ascension 2015.5', 'TIC Declination 2015.5',
       'TMag Uncertainty', 'Epoch Uncertainty', 'Orbital Period Uncertainty',
       'Transit Duration (hours) Uncertainty', 'Transit Depth Uncertainty',
       'Surface Gravity Uncertainty', 'Slug', 'Star Radius Uncertainty',
       'Planet Radius Value', 'Planet Radius Uncertainty',
       'Planet Equilibrium Temperature (K) Value',
       'Effective Temperature Uncertainty', 'Effective Stellar Flux Value',
       'Centroid Offset', 'Master', 'SG1a', 'SG1b', 'SG2', 'SG3', 'SG4', 'SG5',
      

In [9]:
## Rename or drop columns
# all_table.drop(columns=['Split'])
all_table = all_table.rename(columns={'filename': 'File',
                                      'Period': 'Per',
                                      'Duration': 'Dur',
                                      'Transit_Depth': 'Depth',
                                      'star_rad': 'SRad',
                                      'star_rad_est': 'SRadEst',
                                      'star_mass': 'SMass',})

In [11]:
## Make label columns
disps = ['e', 'p', 'n', 'b', 't', 'u', 'j']
users = ['mk', 'ch', 'et', 'md', 'as', 'dm', 'Tansu', 'Shishir']
for d in disps:
    all_table[f'disp_{d}'] = 0

# Set all TOIs commented with "CP", "PC", or "KP" with the label 'pu'
tois_planet_mask = (all_table['comment'].str.contains("(PC)", regex=False).astype(bool) | all_table['comment'].str.contains("(CP)", regex=False).astype((bool)) | all_table['comment'].str.contains("(KP)", regex=False).astype((bool)))


# # Print mismatches - not labelled planet by vetters, but is a TOI labelled KP, CP or PC (known planet, confirmed planet, planetary candidate)
print('Mismatches between TOI comment and vetting labels', all_table[tois_planet_mask][(all_table[tois_planet_mask]['Final'] != 'pt') & (~all_table[tois_planet_mask]['Final'].isna())]['TIC ID'])

# print('Mismatches between TOI comment and vetting labels',
#       all_table[tois_planet_mask][(~all_table[tois_planet_mask]['Final'].isin(['pt', 'pu'])) &
#                                     (~all_table[tois_planet_mask]['Final'].isna())]['TIC ID'])


all_table.loc[tois_planet_mask, 'Final'] = 'pu'

if second_f:
    # Correct label 'EB' into 'eb', 'IS' into 'i' and V into 'jj'
    all_table.loc[all_table['Final'] == 'EB', 'Final'] = 'eb'
    all_table.loc[all_table['Final'] == 'IS', 'Final'] = 'i'
    all_table.loc[all_table['Final'] == 'V', 'Final'] = 'jj'

    # if label is O set as 'jj'
    all_table.loc[all_table['Final'] == 'O', 'Final'] = 'jj'

#Note: The new

## Set labels
def set_labels(row):
    a = ~row.isna()
    if row['Final'] == 'i':
        # skip objects labeled as "inside the star"
        return row
    if a['Final']:
        row[f'disp_{row["Final"][0]}'] = 1
        row[f'disp_{row["Final"][1]}'] = 1
    else:
        for user in users:
            if a[user] and row[user]:
                row[f'disp_{row[user][0]}'] += 1
                row[f'disp_{row[user][1]}'] += 1

    return row

all_table = all_table.apply(set_labels, axis=1)

Mismatches between TOI comment and vetting labels Astro ID
966       300217120
1260     1551168743
1261     1551168745
1270     1716106614
1447      391925531
            ...    
10504     341405597
10526     220524097
10535     382521776
10646     257467784
10764     230013442
Name: TIC ID, Length: 125, dtype: int64


In [ ]:
# all_table.shape

(10991, 69)

In [12]:
## Add more B, J, N labels from Astronet-triage into training set to J class


fpath_triage = 'mnt/tess/astronet/tces-v14-all.csv'
# fpath_triage = 'mnt/tess/astronet/tces-v14-all.csv'

table_triage = pd.read_csv(fpath_triage, header=0, low_memory=False).set_index('Astro ID')

In [15]:
## Add more B, J, N labels from Astronet-triage into training set to J class


fpath_triage = 'mnt/tess/astronet/tces-v14-all.csv'
# fpath_triage = 'mnt/tess/astronet/tces-v14-all.csv'

table_triage = pd.read_csv(fpath_triage, header=0, low_memory=False).set_index('Astro ID')

In [26]:
 table_triage.shape,table_triage[(table_triage['Final']=='J') & (table_triage['File'].str.contains('mk_')) ].shape, table_triage[(table_triage['Final']=='B')].shape, table_triage[table_triage['Final']=='E'].shape,table_triage[table_triage['Final']=='N'].shape,

((28943, 32), (1675, 32), (784, 32), (2659, 32), (155, 32))

In [ ]:
## Add more B, J, N labels from Astronet-triage into training set to J class


fpath_triage = 'mnt/tess/astronet/tces-v14-all.csv'
# fpath_triage = 'mnt/tess/astronet/tces-v14-all.csv'

table_triage = pd.read_csv(fpath_triage, header=0, low_memory=False).set_index('Astro ID')

# Adding junk examples taking junk (J), contact binaries (B) and not sure (N) from astronet triage dataset
junk_df = table_triage[(table_triage['Final'] == 'J') & (table_triage['File'].str.contains('mk_'))]

print('Junk examples:', junk_df.shape)

# Add more B examples
b_df = table_triage[(table_triage['Final'] == 'B') & (table_triage['File'].str.contains('mk_'))]
print('B examples:', b_df.shape)
# Add more N examples
n_df = table_triage[(table_triage['Final'] == 'N') & (table_triage['File'].str.contains('mk_'))]
print('N examples:', n_df.shape)


print('Total junk examples added', junk_df.shape[0] + b_df.shape[0] + n_df.shape[0])


more_bjn_labels = pd.concat([junk_df, b_df, n_df], ignore_index=False)
more_bjn_labels['disp_e'] = 0
more_bjn_labels['disp_p'] = 0
more_bjn_labels['disp_n'] = 0
more_bjn_labels['disp_b'] = 0
more_bjn_labels['disp_t'] = 0
more_bjn_labels['disp_u'] = 0
more_bjn_labels['disp_j'] = 1


new_columns = all_table.keys().intersection(more_bjn_labels.keys()) # Get only overlapping columns from all_table and more_bjn_labels: Index(['TIC ID', 'Final', 'Decision', 'mk', 'ch', 'et', 'md', 'as', 'dm', 'Split', 'RA', 'Dec', 'Tmag', 'Epoc', 'Per', 'Dur', 'Depth', 'SRad', 'SMass', 'SRadEst', 'File', 'disp_e', 'disp_p', 'disp_n', 'disp_b', 'disp_t', 'disp_u', 'disp_j'])

all_table = pd.concat([all_table[new_columns], more_bjn_labels[new_columns]]) # Add 1000 Bs, 1000 Js, and 500 Ns into J class of all_table

Junk examples: (1675, 32)
B examples: (70, 32)
N examples: (6, 32)
Total junk examples added 1751


In [ ]:
# continue from "Only use labelled rows in Preprocess-0vetting-new-objects.ipynb"
#Note: still need to treat the cases with multiple votes somehow

In [ ]:
from sklearn.model_selection import StratifiedKFold

n_splits=5 # Number of splits for cross-validation

#example use:# Setting up cross validation
# kf=StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

# # Results for each fold
# fold_accuracies=[]

# #Array for confusion matrices
# sum_conf_matrix=np.zeros((2,2),dtype=int)

# print(f"\nStarting {n_splits}-Fold Cross-Validation with Decision Tree...")
# print(f"Parameters: criterion={criterion}, splitter={splitter}")

# # Cross-validation loop
# for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
#     print(f"\nFold {fold+1}/{n_splits}")
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]